# Jacobi with MPI one-sided RMA and CUDA buffers

This notebook explains [`src/11-jacobi-rma.cu`](../src/11-jacobi-rma.cu). It uses the same distributed five-point Jacobi stencil as the blocking and overlap examples, but replaces matched send/receive halo exchange with MPI one-sided Remote Memory Access (RMA).

The central communication operation is `MPI_Get`: a rank reads a neighboring rank's boundary row from an MPI window into its own device ghost row. The neighbor does not post a matching receive.

The example is an advanced CUDA-aware MPI case. An MPI implementation may transfer the device data directly or use internal host staging. Passing a CUDA device pointer does not prove that GPUDirect was used.

## 1. What the program does

The program follows this sequence:

1. Initialize MPI and select one GPU for each rank.
2. Divide the global grid into horizontal strips.
3. Allocate two device arrays, `u` and `v`, including top and bottom ghost rows.
4. Create an RMA window for each device array.
5. For every iteration, use `MPI_Get` to fill the current array's ghost rows.
6. Complete the RMA reads with `MPI_Win_flush_all`.
7. Run the CUDA Jacobi kernel and write the result into the other array.
8. Repeat while alternating the roles of `u` and `v`.

The stencil calculation is unchanged. The important difference is how the halo rows are obtained.

## 2. The Jacobi stencil

For each interior point, the next value is the average of its four neighbors:

$$
v_{i,j} = \frac{1}{4}(u_{i-1,j} + u_{i+1,j} + u_{i,j-1} + u_{i,j+1})
$$

Each MPI rank owns a horizontal portion of the global grid. A point on the top or bottom of that portion needs one row owned by a neighboring rank. Those extra rows are called ghost rows or halo rows.

## 3. Domain decomposition and neighbors

The command-line defaults are:

```cpp
int nx = argc > 1 ? atoi(argv[1]) : 4096,
    ny = argc > 2 ? atoi(argv[2]) : 4096,
    iters = argc > 3 ? atoi(argv[3]) : 200;
```

The vertical size must divide evenly among the MPI ranks:

```cpp
int rows = ny / size;
int first = rank * rows;
int up = rank ? rank - 1 : MPI_PROC_NULL;
int down = rank < size - 1 ? rank + 1 : MPI_PROC_NULL;
```

`rows` is the number of real rows owned by one rank. `first` is the global row index corresponding to the first local real row. The first rank has no rank above it, and the last rank has no rank below it, so their missing neighbors are represented by `MPI_PROC_NULL`.

## 4. Device buffer layout

Each rank allocates `(rows + 2) * nx` values:

```cpp
size_t bytes = (size_t)(rows + 2) * nx * sizeof(double);
double *u, *v;
CUDA_CHECK(cudaMalloc(&u, bytes));
CUDA_CHECK(cudaMalloc(&v, bytes));
```

The local layout is:

| Local row | Meaning |
|---:|---|
| `0` | top ghost row |
| `1` through `rows` | real rows owned by this rank |
| `rows + 1` | bottom ghost row |

Important pointer offsets are:

- `current`: top ghost row
- `current + nx`: first real row
- `current + rows * nx`: last real row
- `current + (rows + 1) * nx`: bottom ghost row

The ghost rows are part of the same contiguous CUDA allocation.

In [ ]:
# A small conceptual model of the local allocation.
rows = 4
nx = 6
layout = ["top ghost"] + [f"real row {i}" for i in range(1, rows + 1)] + ["bottom ghost"]
for local_row, label in enumerate(layout):
    print(f"local row {local_row}: {label}, element offset {local_row * nx}")

## 5. Initialization and stencil kernels

The `init` kernel converts a local row index to a global row index with `first + y - 1`. It sets the global boundary to `1.0` and the interior to `0.0`. Both `u` and `v` are initialized so either buffer is valid when it becomes the current array.

The `step` kernel starts at `x = 1` and uses `begin` and `end` to avoid writing the fixed left, top, and bottom boundaries. It reads from `u` and writes a new state to `v`; it never updates a point in place.

## 6. What is an MPI window?

An MPI window describes memory that can be accessed by one-sided operations. In this example, every rank contributes its local device allocation to a window. A rank can then issue `MPI_Get` against a neighbor's window without the neighbor calling `MPI_Recv`.

The window is a communication description; it does not allocate or copy the CUDA memory. The MPI implementation must understand the supplied device pointer for the chosen RMA operation.

## 7. Creating windows for `u` and `v`

```cpp
MPI_Win win_u, win_v;
MPI_Aint window_bytes = static_cast<MPI_Aint>(bytes);
MPI_CHECK(MPI_Win_create(
    u,                                  /* local device buffer to expose */
    window_bytes,                       /* exposed allocation size in bytes */
    sizeof(double),                     /* target displacements count doubles */
    MPI_INFO_NULL,                      /* no implementation-specific hints */
    MPI_COMM_WORLD,                     /* all ranks expose matching windows */
    &win_u                              /* handle for the u allocation */
));
```

The arguments mean:

- `u`: the local CUDA allocation being exposed.
- `window_bytes`: the size of that allocation in bytes.
- `sizeof(double)`: the displacement unit for target offsets. A displacement of `nx` means `nx` doubles, or one row.
- `MPI_INFO_NULL`: no extra implementation hints.
- `MPI_COMM_WORLD`: all ranks collectively create corresponding windows.
- `&win_u`: where MPI stores the window handle.

The source creates the same kind of window for `v`. This is required because `u` and `v` are separate allocations.

## 8. Why there are two windows

The solver alternates the pointers conceptually:

```cpp
double *current = (k % 2 == 0) ? u : v;
double *next = (k % 2 == 0) ? v : u;
MPI_Win current_window = (k % 2 == 0) ? win_u : win_v;
```

On even iterations, the halo rows belong to `u`, so remote reads use `win_u`. On odd iterations, they belong to `v`, so remote reads use `win_v`.

A window is attached to one allocation. Creating only one window and then switching the local pointer would expose the wrong buffer after the first iteration.

## 9. Starting RMA access epochs

```cpp
MPI_CHECK(MPI_Win_lock_all(
    0,                                  /* assert no special lock mode */
    win_u                               /* start an access epoch on u */
));
MPI_CHECK(MPI_Win_lock_all(
    0,                                  /* assert no special lock mode */
    win_v                               /* start an access epoch on v */
));
```

`MPI_Win_lock_all` starts a passive-target access epoch. The application rank can issue RMA operations to all target ranks in the window. The target rank does not need to call a matching receive or explicitly enter an operation at the same time.

The epochs remain active during the iteration loop and are closed with `MPI_Win_unlock_all` during cleanup.

## 10. Reading the upper halo with `MPI_Get`

```cpp
if (up != MPI_PROC_NULL) {
  MPI_CHECK(MPI_Get(
      current,                 /* origin: this rank's top ghost row */
      nx,                      /* number of values in the origin row */
      MPI_DOUBLE,              /* datatype of the origin values */
      up,                      /* target rank above this rank */
      rows * nx,               /* target displacement: target's last row */
      nx,                      /* number of values to read remotely */
      MPI_DOUBLE,              /* datatype of the target values */
      current_window           /* window exposing the current buffer */
  ));
}
```

The origin is `current`, which points to this rank's top ghost row. The target rank is `up`. Because the target window uses `sizeof(double)` as its displacement unit, `rows * nx` points to the target's last real row.

The upper neighbor does not execute a matching receive. It only exposes its current device allocation through the RMA window.

## 11. Reading the lower halo

```cpp
if (down != MPI_PROC_NULL) {
  MPI_CHECK(MPI_Get(
      current + (rows + 1) * nx, /* origin: this rank's bottom ghost row */
      nx,                        /* number of values in the origin row */
      MPI_DOUBLE,                /* datatype of the origin values */
      down,                      /* target rank below this rank */
      nx,                        /* target displacement: target's first row */
      nx,                        /* number of values to read remotely */
      MPI_DOUBLE,                /* datatype of the target values */
      current_window             /* window exposing the current buffer */
  ));
}
```

This time the origin is the bottom ghost row, and the target is `down`. The displacement `nx` selects the target's first real row, because target row zero is its top ghost row.

Together, the two reads fill the ghost rows needed by the five-point stencil.

## 12. Is `MPI_Get` blocking?

No. `MPI_Get` starts a remote read and may return before the data has arrived in the origin buffer. The code therefore uses:

```cpp
MPI_CHECK(MPI_Win_flush_all(
    current_window /* window whose outstanding MPI_Get operations finish */
));
```

`MPI_Win_flush_all` completes all outstanding RMA operations issued through the selected window. After this completion point, the two ghost-row reads have finished from MPI's perspective and the origin buffers may be consumed.

The sequence is:

1. `MPI_Get`: initiate the upper and lower remote reads.
2. `MPI_Win_flush_all`: wait for those reads to complete.
3. `cudaDeviceSynchronize`: establish CUDA/MPI ordering for the device buffer.
4. Launch `step`, which reads the newly filled ghost rows.

## 13. Running the stencil

After the RMA completion point, the code launches:

```cpp
step<<<work, block>>>(current, next, nx, begin, end);
CUDA_CHECK(cudaDeviceSynchronize());
```

The kernel reads the current state, including the freshly received ghost rows, and writes the new state into `next`. Synchronizing after the kernel is important because the next iteration may expose that buffer in an RMA window and read its boundary rows remotely.

Unlike the overlap example, this version does not compute interior rows while the RMA reads are in flight. It waits at `MPI_Win_flush_all` before launching the complete stencil update.

## 14. CUDA-aware RMA portability

The window buffers are allocated with `cudaMalloc`, so this example requires an MPI implementation that supports the relevant RMA operation with CUDA device memory.

Even when the program accepts the pointer, the actual transport may be:

- GPU peer-to-peer communication on one node,
- GPUDirect RDMA through a compatible network adapter,
- an internal pinned-host staging path, or
- another implementation-specific fallback.

Therefore, successful execution proves that the selected MPI stack handled the buffer, but it does not prove that the transfer was zero-copy or used GPUDirect RDMA. Compare implementations and configurations with controlled measurements.

## 15. RMA compared with point-to-point exchange

| Point-to-point halo exchange | RMA halo exchange |
|---|---|
| Uses matching sends and receives | Uses windows and `MPI_Get`/`MPI_Put` |
| Communication partners participate explicitly | The target exposes memory but need not post a receive |
| `MPI_Sendrecv` is straightforward for symmetric halos | Access epochs and flushes must be understood |
| Often the clearest choice for regular stencils | More natural for asymmetric or irregular ownership |

For a regular Jacobi halo, point-to-point communication is usually easier to explain and validate. RMA becomes more attractive when one-sided ownership or irregular updates are central to the algorithm.

## 16. Cleanup and final result

After the iterations, the program selects the buffer containing the final result based on the parity of `iters`, copies one sample value from the GPU, and reduces that sample across ranks.

The windows are then closed and freed before the CUDA allocations are released:

```cpp
MPI_CHECK(MPI_Win_unlock_all(win_u));
MPI_CHECK(MPI_Win_unlock_all(win_v));
MPI_CHECK(MPI_Win_free(&win_u));
MPI_CHECK(MPI_Win_free(&win_v));
CUDA_CHECK(cudaFree(u));
CUDA_CHECK(cudaFree(v));
```

The RMA epochs must end before their windows are freed, and the windows must be freed before their underlying device buffers are released.

## 17. Summary

This example combines:

- horizontal MPI domain decomposition,
- CUDA device allocations with ghost rows,
- MPI windows exposing the alternating `u` and `v` buffers,
- one-sided `MPI_Get` operations for the halo rows,
- `MPI_Win_flush_all` as the RMA completion point, and
- CUDA synchronization before and after the stencil kernel.

The key correctness rule is that the stencil must not read the ghost rows until the corresponding RMA operations have completed. The key portability rule is that CUDA-aware RMA support and the actual direct transport path depend on the MPI implementation, transport, and hardware.